# Imports

In [2]:
from pyspark.sql import functions as F
from presyo_ph.spark import get_spark_session
from presyo_ph.config import BRONZE_PATH

In [3]:
spark = get_spark_session()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/01 20:10:42 WARN Utils: Your hostname, Miggys-TUF-A15.local, resolves to a loopback address: 127.0.0.1; using 10.22.35.70 instead (on interface feth4322)
26/11/01 20:10:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/miggyfortuna/Development/ml/presyo-ph/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/miggyfortuna/.ivy2.5.2/cache
The jars for the packages stored in: /Users/miggyfortuna/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-eaa72715-dac5-41ad-a9da-b33c2ea9fb74;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found o

# Read from Bronze Table

In [4]:
df = spark.read.format("delta").load(str(BRONZE_PATH))

df.printSchema()

root
 |-- ISO3: string (nullable = true)
 |-- country: string (nullable = true)
 |-- adm1_name: string (nullable = true)
 |-- adm2_name: string (nullable = true)
 |-- mkt_name: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- lon: string (nullable = true)
 |-- geo_id: string (nullable = true)
 |-- price_date: string (nullable = true)
 |-- year: string (nullable = true)
 |-- month: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- components: string (nullable = true)
 |-- start_dense_data: string (nullable = true)
 |-- last_survey_point: string (nullable = true)
 |-- data_coverage: string (nullable = true)
 |-- data_coverage_recent: string (nullable = true)
 |-- index_confidence_score: string (nullable = true)
 |-- spatially_interpolated: string (nullable = true)
 |-- beans: string (nullable = true)
 |-- cabbage: string (nullable = true)
 |-- carrots: string (nullable = true)
 |-- eggs: string (nullable = true)
 |-- garlic: string (nullable = true)
 |

In [5]:
df.select(df.components).show(1, truncate=False)

26/11/01 20:10:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|components                                                                                                                                                                                                                                                                                                 |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|beans (1 KG, Index Weight = 1), cabbage (1 KG, Index Weight = 1), carrots (1 KG, Index Weight

In [6]:
df.show(1, truncate=False)

+----+-----------+---------+---------+--------+-----+------+-----------------------+----------+----+-----+--------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+-----------------+-------------+--------------------+----------------------+----------------------+-----+-------+-------+----+------+------+--------+-----+--------+-------+-------+-------+-------+---------------+-----------+---------+---------+---------+---------+-----------------+-------------+---------+---------+---------+---------+-----------------+-------------+------+------+------+------+--------------+----------+--------+--------+--------+--------+----------------+------------+--------+--------+--------+--------+----------------+------------+----------+---

# Silver Transformations
1. Set appropriate data types.
2. Trim white spaces (for safety)
4. Linear interpolation for missing values

## Type casting

In [7]:
# Mappings
metadata_columns = {
    "lat",
    "lon",
    "year",
    "month",
    "data_coverage",
    "data_coverage_recent",
    "index_confidence_score",
}

# Collecting numeric columns
numeric_columns = [
    c for c in df.columns
    if (
        c in metadata_columns
        or c in {"beans", "cabbage", "carrots", "eggs",
                 "garlic", "onions", "potatoes", "rice", "tomatoes"}
        or c.startswith(("o_", "h_", "l_", "c_", "inflation_", "trust_"))
    )
]

# Type casting
for column in numeric_columns:
    df = df.withColumn(column, F.col(column).cast("DOUBLE"))

In [8]:
# Date columns
date_mapping = {
    "price_date": "yyyy-MM-dd",
    "start_dense_data": "MMM yyyy",
    "last_survey_point": "MMM yyyy",
    "_ingested_at": "yyyy-MM-dd"
}

# Type casting: dates
for column, date_format in date_mapping.items():
    df = df.withColumn(
        column,
        F.to_date(F.col(column), date_format)
    )

# Type casting: numeric columns
for column in numeric_columns:
    df = df.withColumn(
        column,
        F.col(column).cast("DOUBLE")
    )

In [9]:
df = df.withColumn(
    "spatially_interpolated",
    F.col("spatially_interpolated").cast("BOOLEAN")
)

In [10]:
df.printSchema()

root
 |-- ISO3: string (nullable = true)
 |-- country: string (nullable = true)
 |-- adm1_name: string (nullable = true)
 |-- adm2_name: string (nullable = true)
 |-- mkt_name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- geo_id: string (nullable = true)
 |-- price_date: date (nullable = true)
 |-- year: double (nullable = true)
 |-- month: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- components: string (nullable = true)
 |-- start_dense_data: date (nullable = true)
 |-- last_survey_point: date (nullable = true)
 |-- data_coverage: double (nullable = true)
 |-- data_coverage_recent: double (nullable = true)
 |-- index_confidence_score: double (nullable = true)
 |-- spatially_interpolated: boolean (nullable = true)
 |-- beans: double (nullable = true)
 |-- cabbage: double (nullable = true)
 |-- carrots: double (nullable = true)
 |-- eggs: double (nullable = true)
 |-- garlic: double (nullable = true)
 |-- on

## Trim whitespaces

In [11]:
from pyspark.sql.types import StringType

string_columns = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
]

df = df.withColumns({
    col: F.trim(F.col(col))
    for col in string_columns
})

In [12]:
df.select(string_columns).limit(5).toPandas()

,ISO3,country,adm1_name,adm2_name,mkt_name,geo_id,currency,components,_source_file
0,PHL,Philippines,Region VI,Iloilo,Iloilo,gid_1069000001225700000,PHP,"beans (1 KG, Index Weight = 1), cabbage (1 KG,...",/Users/miggyfortuna/Development/ml/presyo-ph/d...
1,PHL,Philippines,Region VI,Iloilo,Iloilo,gid_1069000001225700000,PHP,"beans (1 KG, Index Weight = 1), cabbage (1 KG,...",/Users/miggyfortuna/Development/ml/presyo-ph/d...
2,PHL,Philippines,Region VI,Iloilo,Iloilo,gid_1069000001225700000,PHP,"beans (1 KG, Index Weight = 1), cabbage (1 KG,...",/Users/miggyfortuna/Development/ml/presyo-ph/d...
3,PHL,Philippines,Region VI,Iloilo,Iloilo,gid_1069000001225700000,PHP,"beans (1 KG, Index Weight = 1), cabbage (1 KG,...",/Users/miggyfortuna/Development/ml/presyo-ph/d...
4,PHL,Philippines,Region VI,Iloilo,Iloilo,gid_1069000001225700000,PHP,"beans (1 KG, Index Weight = 1), cabbage (1 KG,...",/Users/miggyfortuna/Development/ml/presyo-ph/d...


## Linear interpolation for each commodity

In [13]:
from pyspark.sql import Window

commodity_columns = [
    "beans",
    "cabbage",
    "carrots",
    "eggs",
    "garlic",
    "onions",
    "potatoes",
    "rice",
    "tomatoes",
]

previous_window = (
    Window
    .partitionBy("geo_id")
    .orderBy("price_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

next_window = (
    Window
    .partitionBy("geo_id")
    .orderBy("price_date")
    .rowsBetween(Window.currentRow, Window.unboundedFollowing)
)

MAX_GAP_MONTHS = 7 # 6 month gap maximum

for column in commodity_columns:

    # Find previous and next non-null observations.
    df = (
        df
        .withColumn(
            f"_prev_{column}",
            F.last(F.col(column), ignorenulls=True).over(previous_window)
        )
        .withColumn(
            f"_prev_date_{column}",
            F.last(
                F.when(F.col(column).isNotNull(), F.col("price_date")),
                ignorenulls=True
            ).over(previous_window)
        )
        .withColumn(
            f"_next_{column}",
            F.first(F.col(column), ignorenulls=True).over(next_window)
        )
        .withColumn(
            f"_next_date_{column}",
            F.first(
                F.when(F.col(column).isNotNull(), F.col("price_date")),
                ignorenulls=True
            ).over(next_window)
        )
    )

    # Calculate the distance between observations and current date.
    total_months = F.months_between(
        F.col(f"_next_date_{column}"),
        F.col(f"_prev_date_{column}")
    )

    elapsed_months = F.months_between(
        F.col("price_date"),
        F.col(f"_prev_date_{column}")
    )

    # Calculate interpolated value.
    interpolated_value = (
        F.col(f"_prev_{column}")
        + (
            (F.col(f"_next_{column}") - F.col(f"_prev_{column}"))
            * elapsed_months / total_months
        )
    )

    # Preserve original values and only fill eligible NULLs.
    df = (
        df
        .withColumn(
            f"{column}_filled",
            F.when(
                F.col(column).isNotNull(),
                F.col(column)
            ).when(
                F.col(f"_prev_{column}").isNotNull()
                & F.col(f"_next_{column}").isNotNull()
                & (total_months <= MAX_GAP_MONTHS),
                interpolated_value
            ).otherwise(F.lit(None).cast("DOUBLE"))
        )
        .withColumn(
            f"{column}_was_imputed",
            F.col(column).isNull()
            & F.col(f"{column}_filled").isNotNull()
        )
    )

    # Remove temporary columns.
    df = df.drop(
        f"_prev_{column}",
        f"_prev_date_{column}",
        f"_next_{column}",
        f"_next_date_{column}"
    )

In [14]:
print("Remaining nulls")
df.select([
    F.count(
        F.when(F.col(f"{c}_filled").isNull(), 1)
    ).alias(c)
    for c in commodity_columns
]).show()

Remaining nulls


[Stage 19:>                                                         (0 + 1) / 1]

+-----+-------+-------+-----+------+------+--------+-----+--------+
|beans|cabbage|carrots| eggs|garlic|onions|potatoes| rice|tomatoes|
+-----+-------+-------+-----+------+------+--------+-----+--------+
|20472|  18539|  18225|18591| 21204| 18456|   21633|17301|   17761|
+-----+-------+-------+-----+------+------+--------+-----+--------+



In [15]:
print("No. of imputed values")
df.select([
    F.count(
        F.when(F.col(f"{c}_was_imputed"), 1)
    ).alias(c)
    for c in commodity_columns
]).show()

No. of imputed values


[Stage 27:>                                                         (0 + 1) / 1]

+-----+-------+-------+----+------+------+--------+----+--------+
|beans|cabbage|carrots|eggs|garlic|onions|potatoes|rice|tomatoes|
+-----+-------+-------+----+------+------+--------+----+--------+
| 1470|   2746|   2957|  83|   127|    95|      99|  58|      77|
+-----+-------+-------+----+------+------+--------+----+--------+



In [16]:
for c in commodity_columns:
    print(f"========== {c} ===========")
    df.select(
        F.count("*").alias("total"),
        F.count(c).alias("original_non_null"),
        F.count(f"{c}_filled").alias("filled_non_null"),
        F.count(F.when(F.col(c).isNull(), 1)).alias("original_null"),
        F.count(F.when(F.col(f"{c}_filled").isNull(), 1)).alias("remaining_null"),
        F.count(F.when(F.col(f"{c}_was_imputed"), 1)).alias("imputed")
    ).show()

========== beans ===========
+-----+-----------------+---------------+-------------+--------------+-------+
|total|original_non_null|filled_non_null|original_null|remaining_null|imputed|
+-----+-----------------+---------------+-------------+--------------+-------+
|24743|             2801|           4271|        21942|         20472|   1470|
+-----+-----------------+---------------+-------------+--------------+-------+

========== cabbage ===========
+-----+-----------------+---------------+-------------+--------------+-------+
|total|original_non_null|filled_non_null|original_null|remaining_null|imputed|
+-----+-----------------+---------------+-------------+--------------+-------+
|24743|             3458|           6204|        21285|         18539|   2746|
+-----+-----------------+---------------+-------------+--------------+-------+

========== carrots ===========
+-----+-----------------+---------------+-------------+--------------+-------+
|total|original_non_null|filled_non_nu

In [17]:
df.printSchema()

root
 |-- ISO3: string (nullable = true)
 |-- country: string (nullable = true)
 |-- adm1_name: string (nullable = true)
 |-- adm2_name: string (nullable = true)
 |-- mkt_name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- geo_id: string (nullable = true)
 |-- price_date: date (nullable = true)
 |-- year: double (nullable = true)
 |-- month: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- components: string (nullable = true)
 |-- start_dense_data: date (nullable = true)
 |-- last_survey_point: date (nullable = true)
 |-- data_coverage: double (nullable = true)
 |-- data_coverage_recent: double (nullable = true)
 |-- index_confidence_score: double (nullable = true)
 |-- spatially_interpolated: boolean (nullable = true)
 |-- beans: double (nullable = true)
 |-- cabbage: double (nullable = true)
 |-- carrots: double (nullable = true)
 |-- eggs: double (nullable = true)
 |-- garlic: double (nullable = true)
 |-- on

In [18]:
for c in commodity_columns:
    changed = df.filter(
        F.col(c).isNotNull()
        & (F.col(c) != F.col(f"{c}_filled"))
    ).count()

    print(f"{c}: {changed} changed values")

beans: 0 changed values
cabbage: 0 changed values
carrots: 0 changed values
eggs: 0 changed values
garlic: 0 changed values
onions: 0 changed values
potatoes: 0 changed values
rice: 0 changed values
tomatoes: 0 changed values


In [19]:
df.select(
    *[
        F.col(c).alias(f"{c}_original")
        for c in commodity_columns
    ],
    *[
        F.col(f"{c}_filled").alias(f"{c}_filled")
        for c in commodity_columns
    ]
).describe().toPandas()

,summary,beans_original,cabbage_original,carrots_original,eggs_original,garlic_original,onions_original,potatoes_original,rice_original,tomatoes_original,beans_filled,cabbage_filled,carrots_filled,eggs_filled,garlic_filled,onions_filled,potatoes_filled,rice_filled,tomatoes_filled
0,count,2801,3458,3561,6069,3412,6192,3011,7384,6905,4271,6204,6518,6152,3539,6287,3110,7442,6982
1,mean,99.08767940021457,74.23006362059044,96.2139511373214,7.257246663371168,171.0301817116058,134.3158187984499,38.44476917967468,40.032493228602455,66.85443881245544,99.34708850386374,74.50741134751806,96.34294722307484,7.237848667100069,171.56283413393598,133.43489899793255,38.08996623794227,40.009260279494846,66.62162918934469
2,stddev,15.999337833575902,31.67797893996939,40.728011538824255,1.6123701774961603,58.8726999214279,65.81032931284929,16.007038201934783,6.396141766733846,33.783648953654435,15.264327705063543,30.850931687699035,38.73123973049345,1.6212699059476594,58.93284002813117,65.74268477933934,15.928886525110546,6.390911627617284,33.70831497968227
3,min,55.49,7.52,14.2,3.99,80.91,24.5,10.0,20.0,7.45,55.49,7.52,14.2,3.99,80.91,24.5,10.0,20.0,7.45
4,max,230.77,198.75,300.0,25.0,594.44,617.5,110.0,60.29,285.0,230.77,198.75,300.0,25.0,594.44,617.5,110.0,60.29,285.0


In [20]:
# check uniqueness across geo_id and price_date
df.groupBy(
    "geo_id",
    "price_date"
).count().filter(
    F.col("count") > 1
).show(50, truncate=False)

+------+----------+-----+
|geo_id|price_date|count|
+------+----------+-----+
+------+----------+-----+



In [21]:
# check for negative prices
for c in commodity_columns:
    df.filter(
        F.col(f"{c}_filled") < 0
    ).select(
        "geo_id", "price_date", f"{c}_filled"
    ).show()

+------+----------+------------+
|geo_id|price_date|beans_filled|
+------+----------+------------+
+------+----------+------------+

+------+----------+--------------+
|geo_id|price_date|cabbage_filled|
+------+----------+--------------+
+------+----------+--------------+

+------+----------+--------------+
|geo_id|price_date|carrots_filled|
+------+----------+--------------+
+------+----------+--------------+

+------+----------+-----------+
|geo_id|price_date|eggs_filled|
+------+----------+-----------+
+------+----------+-----------+

+------+----------+-------------+
|geo_id|price_date|garlic_filled|
+------+----------+-------------+
+------+----------+-------------+

+------+----------+-------------+
|geo_id|price_date|onions_filled|
+------+----------+-------------+
+------+----------+-------------+

+------+----------+---------------+
|geo_id|price_date|potatoes_filled|
+------+----------+---------------+
+------+----------+---------------+

+------+----------+-----------+
|geo_

# Notes
Linear interpolation was applied only when the distance between valid bounding observations is at most seven months (six missing months).

In [22]:
from presyo_ph.medallion.silver import save_silver

save_silver(df)

26/11/01 20:13:48 WARN DAGScheduler: Broadcasting large task binary with size 1087.7 KiB
                                                                                

Silver transformation completed: local
